# Lesson 7 — The Complete Analysis Tool (Capstone)

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lessons 1–6  
**You will learn:**
- Organise code into a **class** (`BeamAnalysis`) — Python's way of grouping data and functions together
- Save and load beam configurations to/from **JSON files**
- Handle errors gracefully with **`try/except`**
- Build a **tabbed interface** using `widgets.Tab`
- Export a **PDF report** directly from Python
- Understand the **balanced cantilever** concept by optimising support positions

**Time estimate:** 90 minutes

---

## 7.1 Introducing Classes

So far we have kept geometry and loads in separate dictionaries. A **class** bundles them together with the functions that work on them — just like an engineering object in the real world has both properties and behaviour.

```
BeamAnalysis
  ├── Properties (data)
  │     ├── geometry  (L, x_A, x_B)
  │     └── loads     (list of dicts)
  └── Methods (functions)
        ├── add_point_load()
        ├── add_udl()
        ├── solve()    → returns R_A, R_B, x, V, M
        ├── save()     → writes to JSON file
        └── load()     → reads from JSON file
```

In [ ]:
import json
import numpy as np
import sys
sys.path.insert(0, "../shared")
import beam_solver


class BeamAnalysis:
    """Complete beam analysis: geometry, loads, solver, file I/O."""

    def __init__(self, L_total=10.0, x_A=2.0, x_B=8.0):
        """Create a beam with given geometry. Loads start empty."""
        self.geometry = {"L_total": L_total, "x_A": x_A, "x_B": x_B}
        self.loads    = []

    # ── Load management ────────────────────────────────────────────────────

    def add_point_load(self, x, magnitude):
        L = self.geometry["L_total"]
        if not (0 <= x <= L):
            raise ValueError(f"Position {x} m is outside the beam (0–{L} m).")
        self.loads.append({"type": "point", "x": x, "magnitude": magnitude})
        return self

    def add_udl(self, x1, x2, w):
        if x2 <= x1:
            raise ValueError("UDL end x2 must be greater than start x1.")
        self.loads.append({"type": "udl", "x1": x1, "x2": x2, "w": w})
        return self

    def clear_loads(self):
        self.loads.clear()
        return self

    # ── Solver ────────────────────────────────────────────────────────────

    def solve(self, n_points=500):
        """Run the full analysis and return a results dict."""
        if not self.loads:
            raise ValueError("No loads defined. Add at least one load before solving.")
        return beam_solver.analyse(self.geometry, self.loads, n_points)

    # ── File I/O ──────────────────────────────────────────────────────────

    def save(self, filepath):
        """Save beam configuration to a JSON file."""
        data = {"geometry": self.geometry, "loads": self.loads}
        with open(filepath, "w") as f:
            json.dump(data, f, indent=2)
        print(f"Saved to {filepath}")

    @classmethod
    def load(cls, filepath):
        """Load a beam configuration from a JSON file."""
        with open(filepath, "r") as f:
            data = json.load(f)
        geo = data["geometry"]
        beam = cls(geo["L_total"], geo["x_A"], geo["x_B"])
        beam.loads = data["loads"]
        print(f"Loaded from {filepath}: {len(beam.loads)} load(s)")
        return beam

    def __repr__(self):
        geo = self.geometry
        return (f"BeamAnalysis(L={geo['L_total']} m, "
                f"A={geo['x_A']} m, B={geo['x_B']} m, "
                f"{len(self.loads)} load(s))")


# ── Test the class ─────────────────────────────────────────────────────────
beam = BeamAnalysis(L_total=10.0, x_A=2.0, x_B=8.0)
beam.add_point_load(5.0, -20.0).add_udl(2.0, 8.0, -5.0)

print(beam)

results = beam.solve()
print(f"R_A = {results['R_A']:+.2f} kN,  R_B = {results['R_B']:+.2f} kN")
print(f"M_max = {results['M'].max():+.2f} kN·m at x = {results['x'][results['M'].argmax()]:.2f} m")

Notice the **method chaining**: `beam.add_point_load(...).add_udl(...)` works because each method returns `self`.

---

## 7.2 Saving and Loading with JSON

In [ ]:
import os

# Save the current beam to a file
save_path = "../../my_beam.json"
beam.save(save_path)

# Look at what was saved
with open(save_path) as f:
    print(f.read())

In [ ]:
# Load it back
beam2 = BeamAnalysis.load(save_path)
print(beam2)

results2 = beam2.solve()
print(f"R_A = {results2['R_A']:+.2f} kN  (should match above)")

---

## 7.3 Error Handling with `try / except`

In [ ]:
# What happens if we try to solve an empty beam?
empty_beam = BeamAnalysis()

try:
    results = empty_beam.solve()
except ValueError as e:
    print(f"Caught an error: {e}")

# What about an invalid support position?
try:
    bad_beam = BeamAnalysis(L_total=10.0, x_A=9.0, x_B=2.0)  # A > B!
    bad_beam.add_point_load(5.0, -20.0)
    bad_beam.solve()
except ValueError as e:
    print(f"Caught another error: {e}")

print("Program continues normally after handling the errors.")

Without `try/except`, these errors would crash the entire program. With it, we catch the problem, inform the user, and carry on.

---

## 7.4 The Balanced Cantilever — An Engineering Insight

One beautiful structural result: for a beam carrying a **uniform full-span load**, the peak bending moment is minimised when the cantilever length equals about **20.7% of the total span**. Let's verify this computationally:

In [ ]:
import matplotlib.pyplot as plt

L = 10.0
w = -10.0  # kN/m full-span UDL

# Vary the cantilever length (symmetric: a = L - b)
cant_fractions = np.linspace(0.01, 0.45, 80)
M_peaks = []

for frac in cant_fractions:
    x_A = frac * L
    x_B = L - x_A
    geo   = {"L_total": L, "x_A": x_A, "x_B": x_B}
    loads = [{"type": "udl", "x1": 0.0, "x2": L, "w": w}]
    try:
        res = beam_solver.analyse(geo, loads)
        M_peaks.append(res["M"].max())  # sagging peak (most critical)
    except Exception:
        M_peaks.append(np.nan)

M_peaks = np.array(M_peaks)
optimal_idx = np.nanargmin(M_peaks)
optimal_frac = cant_fractions[optimal_idx]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(cant_fractions * 100, M_peaks, color="#1B5E20", lw=2.2)
ax.axvline(optimal_frac * 100, color="red", ls="--", lw=1.2,
           label=f"Optimal at {optimal_frac*100:.1f}% (≈ 20.7%)")
ax.scatter([optimal_frac * 100], [M_peaks[optimal_idx]],
           color="red", zorder=5, s=60)
ax.annotate(f"Min M = {M_peaks[optimal_idx]:.1f} kN·m",
            xy=(optimal_frac * 100, M_peaks[optimal_idx]),
            xytext=(optimal_frac * 100 + 1.5, M_peaks[optimal_idx] + 3),
            fontsize=9, color="red",
            arrowprops=dict(arrowstyle="->", color="red"))

ax.set_xlabel("Cantilever length as % of total span", fontsize=10)
ax.set_ylabel("Peak sagging moment (kN·m)", fontsize=10)
ax.set_title("Balanced Cantilever — Effect of Support Position on Peak Moment",
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Optimal cantilever fraction: {optimal_frac:.3f} ({optimal_frac*100:.1f}%)")
print(f"Theoretical optimum:          0.207 (20.7%)")

This is the power of Python for structural engineering: a calculation that would take hours by hand was done in seconds across 80 support positions.

---

## 7.5 The Full Capstone Widget — Tabbed Interface

Now we build the complete tool with four tabs:

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import json, os
import sys
sys.path.insert(0, "../shared")
import beam_solver
from plot_helpers import draw_beam, draw_loads, draw_reactions

%matplotlib inline

# ── Shared state ──────────────────────────────────────────────────────────
state = {
    "geometry": {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0},
    "loads": []
}

# ── Output areas ──────────────────────────────────────────────────────────
out_results  = widgets.Output()
out_tab3     = widgets.Output()   # results tab
out_tab4     = widgets.Output()   # optimise tab
out_status   = widgets.Output()

# ═══════════════════════════════════════════════════════════════════════════
# TAB 1 — Geometry
# ═══════════════════════════════════════════════════════════════════════════
style  = {"description_width": "170px"}
layout = widgets.Layout(width="450px")

sl_L  = widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=0.5,
                             description="Total length L (m):", style=style, layout=layout)
sl_xA = widgets.FloatSlider(value=2.0,  min=0.0, max=9.5,  step=0.5,
                             description="Support A at (m):",   style=style, layout=layout)
sl_xB = widgets.FloatSlider(value=8.0,  min=0.5, max=10.0, step=0.5,
                             description="Support B at (m):",   style=style, layout=layout)
out_geo_preview = widgets.Output()

def update_geo(change=None):
    L = sl_L.value
    x_A = min(sl_xA.value, sl_xB.value - 0.5)
    x_B = min(sl_xB.value, L)
    state["geometry"] = {"L_total": L, "x_A": x_A, "x_B": x_B}
    with out_geo_preview:
        out_geo_preview.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 3.0))
        draw_beam(ax, state["geometry"])
        draw_loads(ax, state["loads"], L)
        ax.set_title(
            f"L={L:.1f} m | Cantilever left={x_A:.1f} m | "
            f"Span={x_B-x_A:.1f} m | Cantilever right={L-x_B:.1f} m",
            fontsize=9
        )
        plt.tight_layout(); plt.show()

for sl in [sl_L, sl_xA, sl_xB]:
    sl.observe(update_geo, names="value")

tab1_content = widgets.VBox([
    widgets.HTML("<h3>Beam Geometry</h3>"),
    sl_L, sl_xA, sl_xB,
    widgets.HTML("<i>Preview updates live:</i>"),
    out_geo_preview
])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 2 — Loads
# ═══════════════════════════════════════════════════════════════════════════
dd_type = widgets.Dropdown(
    options=[("Point load","point"),("UDL","udl")],
    description="Load type:", style=style, layout=layout)
sl_x    = widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5,
                               description="Position x (m):", style=style, layout=layout)
sl_P    = widgets.FloatSlider(value=-20.0, min=-200.0, max=200.0, step=5.0,
                               description="Magnitude (kN):", style=style, layout=layout)
sl_x1   = widgets.FloatSlider(value=2.0, min=0.0, max=9.5, step=0.5,
                               description="UDL start x1 (m):", style=style, layout=layout)
sl_x2   = widgets.FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5,
                               description="UDL end x2 (m):",   style=style, layout=layout)
sl_w    = widgets.FloatSlider(value=-5.0, min=-50.0, max=50.0, step=1.0,
                               description="Intensity (kN/m):", style=style, layout=layout)

btn_add_load   = widgets.Button(description="Add Load",  button_style="primary")
btn_clear_load = widgets.Button(description="Clear All", button_style="danger")
out_load_table = widgets.Output()

pt_ctrl2  = widgets.VBox([sl_x, sl_P])
udl_ctrl2 = widgets.VBox([sl_x1, sl_x2, sl_w])
udl_ctrl2.layout.display = "none"

def on_type2(c):
    if c["new"]=="point":
        pt_ctrl2.layout.display=""; udl_ctrl2.layout.display="none"
    else:
        pt_ctrl2.layout.display="none"; udl_ctrl2.layout.display=""
dd_type.observe(on_type2, names="value")

def refresh_load_table():
    with out_load_table:
        out_load_table.clear_output(wait=True)
        if not state["loads"]:
            print("No loads yet. Use the controls above to add some.")
            return
        print(f"{'#':<4} {'Type':<10} {'Details':>40}")
        print("-" * 55)
        for i, ld in enumerate(state["loads"], 1):
            if ld["type"] == "point":
                print(f"{i:<4} {'point':<10} {ld['magnitude']:+.1f} kN  at  x = {ld['x']:.1f} m")
            else:
                print(f"{i:<4} {'UDL':<10} {ld['w']:+.1f} kN/m  from {ld['x1']:.1f} to {ld['x2']:.1f} m")

def on_add_load(b):
    with out_status:
        out_status.clear_output(wait=True)
        try:
            if dd_type.value == "point":
                state["loads"].append({"type":"point","x":sl_x.value,"magnitude":sl_P.value})
                print(f"✓ Added {sl_P.value:+.0f} kN at x={sl_x.value:.1f} m")
            else:
                if sl_x2.value <= sl_x1.value:
                    raise ValueError("x2 must be > x1")
                state["loads"].append({"type":"udl","x1":sl_x1.value,
                                       "x2":sl_x2.value,"w":sl_w.value})
                print(f"✓ Added UDL {sl_w.value:+.0f} kN/m  [{sl_x1.value:.1f}–{sl_x2.value:.1f} m]")
        except ValueError as e:
            print(f"Error: {e}")
    refresh_load_table()
    update_geo()

def on_clear_load(b):
    state["loads"].clear()
    with out_status:
        out_status.clear_output(wait=True); print("All loads cleared.")
    refresh_load_table()
    update_geo()

btn_add_load.on_click(on_add_load)
btn_clear_load.on_click(on_clear_load)

tab2_content = widgets.VBox([
    widgets.HTML("<h3>Applied Loads</h3>"),
    dd_type, pt_ctrl2, udl_ctrl2,
    widgets.HBox([btn_add_load, btn_clear_load]),
    out_status,
    widgets.HTML("<b>Current loads:</b>"),
    out_load_table
])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 3 — Results
# ═══════════════════════════════════════════════════════════════════════════
btn_solve   = widgets.Button(description="Run Analysis", button_style="success",
                              layout=widgets.Layout(width="160px", height="36px"))
btn_pdf     = widgets.Button(description="Export PDF",   button_style="warning",
                              layout=widgets.Layout(width="160px", height="36px"))
btn_save_j  = widgets.Button(description="Save JSON",    button_style="info",
                              layout=widgets.Layout(width="160px", height="36px"))
out_analysis = widgets.Output()

def run_analysis(b=None):
    with out_analysis:
        out_analysis.clear_output(wait=True)
        geo   = state["geometry"]
        loads = state["loads"]

        if not loads:
            print("No loads defined. Go to Tab 2 and add some loads first.")
            return

        try:
            res = beam_solver.analyse(geo, loads)
        except Exception as e:
            print(f"Analysis error: {e}"); return

        R_A, R_B = res["R_A"], res["R_B"]
        x, V, M  = res["x"],   res["V"],  res["M"]

        # Summary
        print("━" * 55)
        print(" ANALYSIS RESULTS")
        print("━" * 55)
        print(f"  Beam length         : {geo['L_total']:.1f} m")
        print(f"  Support A at        : {geo['x_A']:.1f} m")
        print(f"  Support B at        : {geo['x_B']:.1f} m")
        print(f"  Number of loads     : {len(loads)}")
        print("─" * 55)
        print(f"  Reaction R_A        : {R_A:>+10.2f} kN")
        print(f"  Reaction R_B        : {R_B:>+10.2f} kN")
        print(f"  Equilibrium check   : {R_A+R_B+sum(ld['magnitude'] if ld['type']=='point' else ld['w']*(ld['x2']-ld['x1']) for ld in loads):.2e} kN (≈ 0)")
        print("─" * 55)
        print(f"  V_max               : {V.max():>+10.2f} kN  at x = {x[V.argmax()]:.2f} m")
        print(f"  V_min               : {V.min():>+10.2f} kN  at x = {x[V.argmin()]:.2f} m")
        print("─" * 55)
        print(f"  M_max (sagging)     : {M.max():>+10.2f} kN·m at x = {x[M.argmax()]:.2f} m")
        print(f"  M_min (hogging)     : {M.min():>+10.2f} kN·m at x = {x[M.argmin()]:.2f} m")
        print("━" * 55)

        # Three-panel figure
        fig = plt.figure(figsize=(14, 10))
        gs_ = fig.add_gridspec(3, 1, height_ratios=[1, 1.8, 1.8], hspace=0.08)
        ax_b = fig.add_subplot(gs_[0])
        ax_s = fig.add_subplot(gs_[1])
        ax_m = fig.add_subplot(gs_[2])

        draw_beam(ax_b, geo)
        draw_loads(ax_b, loads, geo["L_total"])
        draw_reactions(ax_b, geo["x_A"], geo["x_B"], R_A, R_B)

        ax_s.plot(x, V, "#1565C0", lw=2)
        ax_s.fill_between(x, V, 0, where=(V>=0), color="#90CAF9", alpha=0.5)
        ax_s.fill_between(x, V, 0, where=(V<0),  color="#EF9A9A", alpha=0.5)
        ax_s.axhline(0, color="k", lw=0.8)
        ax_s.set_ylabel("V (kN)", fontsize=9)
        ax_s.grid(True, ls="--", alpha=0.35)

        ax_m.plot(x, M, "#1B5E20", lw=2)
        ax_m.fill_between(x, M, 0, where=(M>=0), color="#A5D6A7", alpha=0.5, label="Sagging (+)")
        ax_m.fill_between(x, M, 0, where=(M<0),  color="#FFCDD2", alpha=0.5, label="Hogging (−)")
        ax_m.axhline(0, color="k", lw=0.8)
        ax_m.set_ylabel("M (kN·m)", fontsize=9)
        ax_m.set_xlabel("Position x (m)", fontsize=10)
        ax_m.legend(fontsize=8, loc="upper right")
        ax_m.grid(True, ls="--", alpha=0.35)

        zc = np.where(np.diff(np.sign(V)))[0]
        for idx in zc:
            ax_s.axvline(x[idx], color="red", ls=":", lw=1, alpha=0.7)
            ax_m.axvline(x[idx], color="red", ls=":", lw=1, alpha=0.7)

        xlim = (x[0]-0.3, x[-1]+0.3)
        for ax_ in [ax_b, ax_s, ax_m]:
            ax_.set_xlim(*xlim)
        ax_b.tick_params(labelbottom=False)
        ax_s.tick_params(labelbottom=False)

        fig.suptitle(
            f"Complete Beam Analysis  |  L={geo['L_total']} m  |  "
            f"R_A={R_A:+.1f} kN  R_B={R_B:+.1f} kN",
            fontsize=10, y=1.01
        )
        plt.tight_layout()
        plt.show()
        state["last_results"] = res   # store for PDF export

def export_pdf(b):
    with out_analysis:
        if "last_results" not in state:
            print("Run the analysis first, then export.")
            return
        res = state["last_results"]
        geo = state["geometry"]
        R_A, R_B = res["R_A"], res["R_B"]
        x, V, M  = res["x"],   res["V"],  res["M"]

        fig = plt.figure(figsize=(14, 10))
        gs_ = fig.add_gridspec(3, 1, height_ratios=[1, 1.8, 1.8], hspace=0.08)
        ax_b = fig.add_subplot(gs_[0])
        ax_s = fig.add_subplot(gs_[1])
        ax_m = fig.add_subplot(gs_[2])
        draw_beam(ax_b, geo)
        draw_loads(ax_b, state["loads"], geo["L_total"])
        draw_reactions(ax_b, geo["x_A"], geo["x_B"], R_A, R_B)
        ax_s.plot(x, V, "#1565C0", lw=2)
        ax_s.fill_between(x, V, 0, where=(V>=0), color="#90CAF9", alpha=0.5)
        ax_s.fill_between(x, V, 0, where=(V<0),  color="#EF9A9A", alpha=0.5)
        ax_s.axhline(0, color="k", lw=0.8); ax_s.set_ylabel("V (kN)", fontsize=9)
        ax_s.grid(True, ls="--", alpha=0.35)
        ax_m.plot(x, M, "#1B5E20", lw=2)
        ax_m.fill_between(x, M, 0, where=(M>=0), color="#A5D6A7", alpha=0.5)
        ax_m.fill_between(x, M, 0, where=(M<0),  color="#FFCDD2", alpha=0.5)
        ax_m.axhline(0, color="k", lw=0.8)
        ax_m.set_ylabel("M (kN·m)", fontsize=9); ax_m.set_xlabel("x (m)", fontsize=10)
        ax_m.grid(True, ls="--", alpha=0.35)
        xlim = (x[0]-0.3, x[-1]+0.3)
        for ax_ in [ax_b, ax_s, ax_m]: ax_.set_xlim(*xlim)
        ax_b.tick_params(labelbottom=False); ax_s.tick_params(labelbottom=False)
        fig.suptitle(
            f"Beam Analysis Report  |  L={geo['L_total']} m  |  "
            f"A@{geo['x_A']} m  B@{geo['x_B']} m  |  "
            f"R_A={R_A:+.1f} kN  R_B={R_B:+.1f} kN  |  "
            f"M_max={M.max():+.1f} kN·m  M_min={M.min():+.1f} kN·m",
            fontsize=9, y=1.01
        )
        plt.tight_layout()
        pdf_path = "../../beam_analysis_report.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        plt.close(fig)
        print(f"✓ PDF saved to {os.path.abspath(pdf_path)}")

def save_json(b):
    data = {"geometry": state["geometry"], "loads": state["loads"]}
    path = "../../beam_config.json"
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    with out_analysis:
        print(f"✓ Configuration saved to {os.path.abspath(path)}")

btn_solve.on_click(run_analysis)
btn_pdf.on_click(export_pdf)
btn_save_j.on_click(save_json)

tab3_content = widgets.VBox([
    widgets.HTML("<h3>Analysis Results</h3>"),
    widgets.HBox([btn_solve, btn_pdf, btn_save_j]),
    widgets.HTML("<i>Click 'Run Analysis' after setting geometry and loads in Tabs 1 & 2.</i>"),
    out_analysis
])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 4 — Optimise (balanced cantilever)
# ═══════════════════════════════════════════════════════════════════════════
sl_cant_frac = widgets.FloatSlider(
    value=0.207, min=0.01, max=0.45, step=0.005,
    description="Cantilever fraction:",
    style=style, layout=layout,
    readout_format=".3f"
)
out_opt = widgets.Output()

def update_optimise(change=None):
    frac = sl_cant_frac.value
    L    = state["geometry"]["L_total"]
    x_A  = frac * L
    x_B  = L - x_A
    geo  = {"L_total": L, "x_A": x_A, "x_B": x_B}

    # Default load: full-span UDL
    loads_opt = [{"type": "udl", "x1": 0.0, "x2": L, "w": -10.0}]

    with out_opt:
        out_opt.clear_output(wait=True)
        try:
            res = beam_solver.analyse(geo, loads_opt)
        except Exception as e:
            print(f"Error: {e}"); return

        x, V, M = res["x"], res["V"], res["M"]

        fig, (ax_b, ax_m) = plt.subplots(2, 1, figsize=(13, 7),
                                          gridspec_kw={"height_ratios": [1, 2.2]})
        fig.subplots_adjust(hspace=0.08)
        draw_beam(ax_b, geo)
        draw_loads(ax_b, loads_opt, L)
        draw_reactions(ax_b, x_A, x_B, res["R_A"], res["R_B"])

        ax_m.plot(x, M, "#1B5E20", lw=2)
        ax_m.fill_between(x, M, 0, where=(M>=0), color="#A5D6A7", alpha=0.5, label="Sagging")
        ax_m.fill_between(x, M, 0, where=(M<0),  color="#FFCDD2", alpha=0.5, label="Hogging")
        ax_m.axhline(0, color="k", lw=0.8)
        ax_m.set_ylabel("M (kN·m)", fontsize=9)
        ax_m.set_xlabel("Position x (m)", fontsize=10)
        ax_m.legend(fontsize=8)
        ax_m.grid(True, ls="--", alpha=0.35)
        ax_m.set_xlim(x[0]-0.3, x[-1]+0.3)
        ax_b.set_xlim(x[0]-0.3, x[-1]+0.3)
        ax_b.tick_params(labelbottom=False)

        M_sag = M.max(); M_hog = M.min()
        fig.suptitle(
            f"Cantilever = {frac*100:.1f}%  |  x_A={x_A:.2f} m  x_B={x_B:.2f} m  |  "
            f"M_max(sag)={M_sag:+.1f}  M_max(hog)={M_hog:+.1f} kN·m  "
            f"[Optimal at 20.7%]",
            fontsize=9.5, y=1.01
        )
        plt.tight_layout(); plt.show()

sl_cant_frac.observe(update_optimise, names="value")

tab4_content = widgets.VBox([
    widgets.HTML(
        "<h3>Balanced Cantilever Optimisation</h3>"
        "<p>Shows how moving supports (symmetrically) changes the BMD under a full-span UDL of 10 kN/m.</p>"
        "<p>The <b>balanced cantilever</b> occurs at ≈ 20.7% — this minimises the peak bending moment.</p>"
    ),
    sl_cant_frac,
    out_opt
])

# ═══════════════════════════════════════════════════════════════════════════
# Assemble tabs
# ═══════════════════════════════════════════════════════════════════════════
tab = widgets.Tab(children=[tab1_content, tab2_content, tab3_content, tab4_content])
tab.set_title(0, "1. Geometry")
tab.set_title(1, "2. Loads")
tab.set_title(2, "3. Results")
tab.set_title(3, "4. Optimise")

# Initial renders
update_geo()
refresh_load_table()
update_optimise()

display(widgets.VBox([
    widgets.HTML("<h2>Complete Beam Analysis Tool</h2>"),
    tab
]))

---

## Summary — The Complete Course

You have now built a fully functional structural analysis tool from scratch using Python. Here is what you learned across all seven lessons:

| Lesson | Python concepts | Structural concepts |
|--------|----------------|---------------------|
| 1 | Variables, arithmetic, f-strings, `if/else`, `ipywidgets` | Cross-section properties (A, I, Z) |
| 2 | NumPy arrays, `matplotlib`, list indexing | Beam geometry, pin vs roller, cantilevers |
| 3 | Dictionaries, functions, `Button` widget | Sign convention, point loads, UDL |
| 4 | Multiple return values, `assert`, `raise` | ΣFy=0, ΣM=0, statically determinate beams |
| 5 | Array loops, `np.where`, module `import` | Shear force diagram, SFD rules |
| 6 | `cumulative_trapezoid`, `fill_between`, `gridspec` | BMD as integral of SFD, sagging/hogging |
| 7 | Classes, JSON file I/O, `try/except`, `Tab` widget | Balanced cantilever, optimal support position |

## What Comes Next

This course is the foundation. Future courses will build on it:
- **Beam Deflection** — integrating twice to get displacement, using EI
- **Multi-span continuous beams** — moment distribution method, matrix stiffness
- **Section design** — checking bending capacity, shear capacity, deflection limits against code

The Python skills you have gained apply directly to all of these.